In [5]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))

from src.utils.logger import logging

ANALYTICS_DIR = Path("data/processed/analytics")
CHARTS_DIR = ANALYTICS_DIR / "charts"
REPORTS_DIR = ANALYTICS_DIR / "reports"

ANALYTICS_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV_PATH = ANALYTICS_DIR / "raw_news_data.csv"
OPTIMIZED_CSV_PATH = ANALYTICS_DIR / "optimized_news_data.csv"

logging.info("Lab 8 News Media Monitoring notebook started")

print("Project root:", PROJECT_ROOT)
print("Analytics output folder:", ANALYTICS_DIR)

Project root: c:\Users\38760\Desktop\Unstructured_Data\News_Media_Monitoring_Pipeline_UD
Analytics output folder: data\processed\analytics


In [6]:
import numpy as np
import pandas as pd

from src.analytics.numpy_ops import (
    demonstrate_array_creation,
    print_array_info,
    vectorized_operations,
    axis_reductions,
    broadcasting_example,
)

logging.info("Lab 8 notebook Part 2 started: NumPy foundations")

# 1. Create NumPy arrays using 4+ methods
arrays = demonstrate_array_creation()
print_array_info(arrays)

array_info_rows = []

for name, arr in arrays.items():
    array_info_rows.append({
        "array_name": name,
        "shape": str(arr.shape),
        "dtype": str(arr.dtype),
        "ndim": arr.ndim,
        "size": arr.size,
        "itemsize": arr.itemsize,
    })

array_info_df = pd.DataFrame(array_info_rows)
array_info_df.to_csv(REPORTS_DIR / "numpy_array_info_news.csv", index=False)

display(array_info_df)

# 2. Vectorized arithmetic — no Python loops for math
rating_scores = np.array([8.2, 7.4, 6.9, 9.1, 5.8, 8.7])
mention_counts = np.array([34, 28, 19, 23, 14, 17])

vectorized_result = vectorized_operations(rating_scores, mention_counts)

vectorized_df = pd.DataFrame({
    "rating_score": rating_scores,
    "mention_count": mention_counts,
    "normalised": vectorized_result["normalised"],
    "weighted": vectorized_result["weighted"],
    "high_rated": vectorized_result["high_rated"],
    "high_impact": vectorized_result["high_impact"],
    "broadcasting_normalised": broadcasting_example(rating_scores),
})

vectorized_df.to_csv(
    REPORTS_DIR / "numpy_vectorized_operations_news.csv",
    index=False,
)

print("\nVectorized operation statistics:")
print(vectorized_result["stats"])

display(vectorized_df)

# 3. Axis reductions
news_matrix = np.array([
    [8.2, 34, 120.4],
    [7.4, 28, 95.7],
    [6.9, 19, 80.2],
    [9.1, 23, 63.1],
])

axis_result = axis_reductions(news_matrix)

axis_df = pd.DataFrame({
    "col_means": axis_result["col_means"],
    "col_stds": axis_result["col_stds"],
})

axis_df.to_csv(
    REPORTS_DIR / "numpy_axis_reductions_news.csv",
    index=False,
)

row_means_df = pd.DataFrame({
    "row_means": axis_result["row_means"],
})

row_means_df.to_csv(
    REPORTS_DIR / "numpy_row_reductions_news.csv",
    index=False,
)

print("\nColumn means:")
print(axis_result["col_means"])

print("\nRow means:")
print(axis_result["row_means"])

print("\nColumn standard deviations:")
print(axis_result["col_stds"])

logging.info("Lab 8 notebook Part 2 complete")

article_mentions                 shape=(6,)         dtype=int64  ndim=1  size=6  itemsize=8
missing_sentiment_placeholder    shape=(6,)         dtype=float64  ndim=1  size=6  itemsize=8
category_weights                 shape=(3, 4)       dtype=float64  ndim=2  size=12  itemsize=8
years                            shape=(7,)         dtype=int64  ndim=1  size=7  itemsize=8
score_buffer                     shape=(2, 3)       dtype=float64  ndim=2  size=6  itemsize=8


,array_name,shape,dtype,ndim,size,itemsize
0,article_mentions,"(6,)",int64,1,6,8
1,missing_sentiment_placeholder,"(6,)",float64,1,6,8
2,category_weights,"(3, 4)",float64,2,12,8
3,years,"(7,)",int64,1,7,8
4,score_buffer,"(2, 3)",float64,2,6,8



Vectorized operation statistics:
{'mean': 7.683333333333333, 'std': 1.1216307572260824, 'max': 9.1, 'min_mentions': 14, 'total_mentions': 135}


,rating_score,mention_count,normalised,weighted,high_rated,high_impact,broadcasting_normalised
0,8.2,34,82.0,29.15,True,True,0.727273
1,7.4,28,74.0,24.92,False,True,0.484848
2,6.9,19,69.0,20.67,False,False,0.333333
3,9.1,23,91.0,28.92,True,True,1.000000
4,5.8,14,58.0,15.71,False,False,0.000000
5,8.7,17,87.0,25.15,True,False,0.878788



Column means:
[ 7.9  26.   89.85]

Row means:
[54.2        43.7        35.36666667 31.73333333]

Column standard deviations:
[ 0.8336666   5.61248608 21.07255324]


In [14]:
from src.analytics.data_loader import (
    load_from_mongodb,
    save_to_csv,
    load_from_csv,
    chunked_stats,
    process_chunks_per_language,
    optimise_dtypes,
    memory_comparison,
)

logging.info("Lab 8 notebook Part 3 started: loading and memory management")

news_df = load_from_mongodb()

print("Loaded integrated news data from MongoDB:")
print(news_df.shape)
display(news_df.head())

if news_df.empty:
    raise ValueError("MongoDB returned an empty dataset. Check news_pipeline.raw_articles.")

save_to_csv(news_df, str(RAW_CSV_PATH))
print("Saved raw news CSV:", RAW_CSV_PATH)

csv_df = load_from_csv(str(RAW_CSV_PATH))

print("Loaded CSV shape:")
print(csv_df.shape)

chunk_results = chunked_stats(
    str(RAW_CSV_PATH),
    chunk_size=100,
    rating_column="rating_score",
    language_column="language",
)

print("Chunked global mean rating_score:")
print(chunk_results["global_mean"])

print("Total rows processed:")
print(chunk_results["total_rows"])

pd.DataFrame([{
    "global_mean_rating_score": chunk_results["global_mean"],
    "total_rows": chunk_results["total_rows"],
    "rating_count": chunk_results["rating_count"],
}]).to_csv(REPORTS_DIR / "chunked_global_rating_score_mean.csv", index=False)

language_stats = process_chunks_per_language(
    str(RAW_CSV_PATH),
    chunk_size=100,
    rating_column="rating_score",
    language_column="language",
)

language_stats.to_csv(REPORTS_DIR / "chunked_language_rating_score_stats.csv", index=False)

print("Per-language chunked statistics:")
display(language_stats)

optimized_df = optimise_dtypes(csv_df)
memory_stats = memory_comparison(csv_df, optimized_df)

pd.DataFrame([memory_stats]).to_csv(REPORTS_DIR / "memory_optimisation_report_news.csv", index=False)

print("Memory optimization report:")
print(memory_stats)

save_to_csv(optimized_df, str(OPTIMIZED_CSV_PATH))

logging.info("Lab 8 notebook Part 3 complete")

Loaded integrated news data from MongoDB:
(1455, 60)


,record_id,source_name,author,title,description,url,publishedAt,source_path,fetched_at,version,...,title_length,rating_score,overview,genres,popularity,release_date,release_year,original_language,vote_average,vote_count
0,1,Motley Fool Australia,James Mickleboro,"Why EOS, Humm, New Hope, and Sims shares are s...",These shares are having a good session on hump...,https://www.fool.com.au/2026/03/18/why-eos-hum...,2026-03-18T02:40:46Z,NaN,NaT,NaN,...,66,1.81,These shares are having a good session on hump...,news_api,181.0,NaT,NaN,unknown,1.81,1.0
1,2,The Punch,Punch Newspapers,SWDC partners FTID to boost S’West rural devt,The South-West Development Commission (SWDC) a...,https://punchng.com/swdc-partners-ftid-to-boos...,2026-03-18T02:34:47Z,NaN,NaT,NaN,...,45,2.26,The South-West Development Commission (SWDC) a...,news_api,226.0,NaT,NaN,unknown,2.26,1.0
2,3,The Times of India,TOI Education,"GATE 2026 final answer key not released yet, r...",The Indian Institute of Technology Guwahati ha...,https://timesofindia.indiatimes.com/education/...,2026-03-18T02:31:14Z,NaN,NaT,NaN,...,98,2.60,The Indian Institute of Technology Guwahati ha...,news_api,260.0,NaT,NaN,unknown,2.60,1.0
3,4,Financial Post,Business Wire,LTM Named NVIDIA Partner Network ‘Rising Star ...,"MUMBAI, India — LTM, the Business Creativity p...",https://financialpost.com/pmn/business-wire-ne...,2026-03-18T02:30:15Z,NaN,NaT,NaN,...,96,2.60,"MUMBAI, India — LTM, the Business Creativity p...",news_api,260.0,NaT,NaN,unknown,2.60,1.0
4,5,CNA,NaN,"Asian stocks rally as oil retreats, Fed in spo...","SYDNEY, March 18 : Asian shares rallied on Wed...",https://www.channelnewsasia.com/business/asian...,2026-03-18T02:27:11Z,NaN,NaT,NaN,...,52,2.60,"SYDNEY, March 18 : Asian shares rallied on Wed...",news_api,260.0,NaT,NaN,unknown,2.60,1.0


Saved raw news CSV: data\processed\analytics\raw_news_data.csv
Loaded CSV shape:
(1455, 60)
Chunked global mean rating_score:
1.1160137457044674
Total rows processed:
1455
Per-language chunked statistics:


,language,mean_rating_score,record_count
0,unknown,1.116014,1455.0


Memory optimization report:
{'before_mb': np.float64(2.8278627395629883), 'after_mb': np.float64(1.4204998016357422), 'reduction_pct': np.float64(49.76772451638642)}


In [15]:
from src.analytics.explorer import (
    inspect_shape,
    print_info,
    describe_numeric,
    describe_all,
    value_counts_report,
    nunique_report,
    extract_release_year,
    plot_distributions,
)

logging.info("Lab 8 notebook Part 4 started: EDA")

eda_df = csv_df.copy()

eda_df = extract_release_year(eda_df)

shape_info = inspect_shape(eda_df)

pd.DataFrame([shape_info]).to_csv(REPORTS_DIR / "eda_shape_report_news.csv", index=False)

print("Shape information:")
print(shape_info)

print("\nDataFrame info:")
print_info(eda_df)

numeric_description = describe_numeric(eda_df)
full_description = describe_all(eda_df)

numeric_description.to_csv(REPORTS_DIR / "eda_numeric_describe_news.csv")
full_description.to_csv(REPORTS_DIR / "eda_full_describe_news.csv")

print("\nNumeric describe:")
display(numeric_description)

print("\nFull describe:")
display(full_description)

counts_report = value_counts_report(
    eda_df,
    cols=["document_type", "category", "source_name", "language"],
    top_n=15,
)

for col, report in counts_report.items():
    print(f"\nColumn: {col}")
    print("Unique values:", report["nunique"])
    display(report["counts"])

    report["counts"].to_csv(
        REPORTS_DIR / f"eda_value_counts_{col}_news.csv",
        header=["count"],
    )

unique_report = nunique_report(eda_df)
unique_report.to_csv(REPORTS_DIR / "eda_nunique_report_news.csv", index=False)

print("\nNunique report:")
display(unique_report)

saved_charts = plot_distributions(
    eda_df,
    output_dir=str(CHARTS_DIR),
)

print("\nSaved charts:")
for chart in saved_charts:
    print(chart)

logging.info("Lab 8 notebook Part 4 complete")

Shape information:
{'rows': 1455, 'columns': 60, 'cells': 87300, 'column_names': ['record_id', 'source_name', 'author', 'title', 'description', 'url', 'publishedAt', 'source_path', 'fetched_at', 'version', 'document_type', 'file_name', 'page_number', 'extraction_timestamp', 'extraction_library', 'text', 'tables', 'id', 'category', 'published_date', 'mentions', 'sentiment_score', 'sheet_name', 'Metric', 'Total Articles', 'Total Mentions', 'Average Mentions', 'Average Sentiment', 'Highest Mentions', 'Politics Articles', 'Business Articles', 'paragraph_number', 'run_number', 'bold', 'italic', 'underline', 'preview_text', 'name', 'year', 'wins', 'losses', 'nominations', 'awards', 'best_picture', 'raw_text', 'processed_text', 'content_text', 'language', 'published_year', 'content_length', 'title_length', 'rating_score', 'overview', 'genres', 'popularity', 'release_date', 'release_year', 'original_language', 'vote_average', 'vote_count']}

DataFrame info:
<class 'pandas.DataFrame'>
RangeInde

,record_id,fetched_at,version,page_number,extraction_timestamp,id,published_date,mentions,sentiment_score,Total Articles,...,awards,published_year,content_length,title_length,rating_score,popularity,release_date,release_year,vote_average,vote_count
count,1455.000000,1361,1361.0,546.000000,618,90.000000,90,90.000000,90.0000,9.0,...,435.000000,90.0,1455.000000,1455.000000,1455.000000,1455.000000,90,90.0,1455.000000,1455.000000
mean,728.000000,2026-04-12 18:01:23.771983,1.0,2.415751,2026-04-11 12:33:08.038402,5.500000,2026-03-22 12:00:00,23.500000,0.5100,10.0,...,1.666667,2026.0,94.283849,21.503093,1.116014,94.580756,2026-03-22 12:00:00,2026.0,1.116014,2.391753
min,1.000000,2026-04-01 02:26:18.663000,1.0,1.000000,2026-04-01 02:10:52.965282,1.000000,2026-03-20 00:00:00,14.000000,0.2000,10.0,...,1.000000,2026.0,3.000000,3.000000,0.030000,3.000000,2026-03-20 00:00:00,2026.0,0.030000,1.000000
25%,364.500000,2026-04-09 15:21:30.718000,1.0,1.000000,2026-04-06 00:28:31.571914,3.000000,2026-03-21 00:00:00,19.000000,0.4000,10.0,...,1.000000,2026.0,15.000000,15.000000,0.150000,15.000000,2026-03-21 00:00:00,2026.0,0.150000,1.000000
50%,728.000000,2026-04-09 15:51:53.588000,1.0,2.000000,2026-04-09 15:51:38.185589,5.500000,2026-03-22 12:00:00,22.500000,0.5000,10.0,...,1.000000,2026.0,15.000000,15.000000,0.150000,15.000000,2026-03-22 12:00:00,2026.0,0.150000,1.000000
75%,1091.500000,2026-04-16 00:34:19.422000,1.0,3.000000,2026-04-16 00:34:12.088912,8.000000,2026-03-24 00:00:00,28.000000,0.7000,10.0,...,2.000000,2026.0,21.000000,16.000000,0.585000,24.000000,2026-03-24 00:00:00,2026.0,0.585000,1.000000
max,1455.000000,2026-04-26 02:17:59.397000,1.0,4.000000,2026-04-26 02:17:53.010339,10.000000,2026-03-25 00:00:00,34.000000,0.8000,10.0,...,7.000000,2026.0,2124.000000,202.000000,10.000000,2124.000000,2026-03-25 00:00:00,2026.0,10.000000,34.000000
std,420.166634,NaN,0.0,1.115878,NaN,2.888373,NaN,5.987346,0.1824,0.0,...,1.257848,0.0,268.337889,21.649670,2.185511,268.260216,NaN,0.0,2.185511,5.620672



Full describe:


,record_id,source_name,author,title,description,url,publishedAt,source_path,fetched_at,version,...,title_length,rating_score,overview,genres,popularity,release_date,release_year,original_language,vote_average,vote_count
count,1455.000000,1455,157,1455,164,164,164,1361,1361,1361.0,...,1455.000000,1455.000000,1455,1455,1455.000000,90,90.0,1455,1455.000000,1455.000000
unique,NaN,80,89,203,107,107,97,22,NaN,NaN,...,NaN,NaN,229,19,NaN,NaN,NaN,1,NaN,NaN
top,NaN,unknown,Reuters,Untitled record,These shares are having a good session on hump...,https://www.fool.com.au/2026/03/18/why-eos-hum...,2026-03-18T02:40:46Z,https://www.scrapethissite.com/pages/forms/,NaN,NaN,...,NaN,NaN,Untitled record,scraped_html_paginated,NaN,NaN,NaN,unknown,NaN,NaN
freq,NaN,1189,16,766,5,5,5,125,NaN,NaN,...,NaN,NaN,634,500,NaN,NaN,NaN,1455,NaN,NaN
mean,728.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-12 18:01:23.771983,1.0,...,21.503093,1.116014,NaN,NaN,94.580756,2026-03-22 12:00:00,2026.0,NaN,1.116014,2.391753
min,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-01 02:26:18.663000,1.0,...,3.000000,0.030000,NaN,NaN,3.000000,2026-03-20 00:00:00,2026.0,NaN,0.030000,1.000000
25%,364.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-09 15:21:30.718000,1.0,...,15.000000,0.150000,NaN,NaN,15.000000,2026-03-21 00:00:00,2026.0,NaN,0.150000,1.000000
50%,728.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-09 15:51:53.588000,1.0,...,15.000000,0.150000,NaN,NaN,15.000000,2026-03-22 12:00:00,2026.0,NaN,0.150000,1.000000
75%,1091.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-16 00:34:19.422000,1.0,...,16.000000,0.585000,NaN,NaN,24.000000,2026-03-24 00:00:00,2026.0,NaN,0.585000,1.000000
max,1455.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-26 02:17:59.397000,1.0,...,202.000000,10.000000,NaN,NaN,2124.000000,2026-03-25 00:00:00,2026.0,NaN,10.000000,34.000000



Column: document_type
Unique values: 15


...


Column: language
Unique values: 1


language
unknown    1455
Name: count, dtype: int64


Nunique report:


,column,nunique,dtype
0,record_id,1455,int64
1,fetched_at,1137,datetime64[us]
2,extraction_timestamp,618,datetime64[us]
3,overview,229,str
4,content_text,229,str
5,title,203,str
6,url,107,str
7,description,107,str
8,publishedAt,97,str
9,popularity,91,float64



Saved charts:
data\processed\analytics\charts\rating_score_distribution.png
data\processed\analytics\charts\rating_distribution.png
data\processed\analytics\charts\popularity_distribution.png
data\processed\analytics\charts\mentions_distribution.png
data\processed\analytics\charts\content_length_distribution.png
data\processed\analytics\charts\language_distribution.png
data\processed\analytics\charts\published_year_distribution.png
data\processed\analytics\charts\release_year_distribution.png
data\processed\analytics\charts\category_distribution.png
data\processed\analytics\charts\genre_distribution.png
data\processed\analytics\charts\document_type_distribution.png
data\processed\analytics\charts\source_distribution.png


In [16]:
from src.analytics.selector import (
    select_columns,
    loc_filter,
    iloc_sample,
    boolean_filter,
    isin_filter,
    between_filter,
)

logging.info("Lab 8 notebook Part 5 started: selection and filtering")

selection_df = eda_df.copy()

# 1. Column selection
selected_columns = select_columns(
    selection_df,
    [
        "record_id",
        "title",
        "document_type",
        "category",
        "source_name",
        "rating_score",
        "language",
        "content_length",
        "published_year",
    ],
)

selected_columns.to_csv(
    REPORTS_DIR / "selection_selected_columns_news.csv",
    index=False,
)

print("Selected columns:")
display(selected_columns.head())

# 2. loc filtering
loc_result = loc_filter(
    selection_df,
    min_rating_score=7.0,
    result_cols=[
        "record_id",
        "title",
        "document_type",
        "category",
        "rating_score",
        "content_length",
    ],
)

loc_result.to_csv(
    REPORTS_DIR / "selection_loc_filter_news.csv",
    index=False,
)

print("loc filter result:")
display(loc_result.head())

# 3. iloc sampling
iloc_result = iloc_sample(
    selection_df,
    step=20,
)

iloc_result.to_csv(
    REPORTS_DIR / "selection_iloc_sample_news.csv",
    index=False,
)

print("iloc sample result:")
display(iloc_result.head())

# 4. Boolean filtering
boolean_result = boolean_filter(
    selection_df,
    min_rating_score=5.0,
    min_mentions=1,
    min_popularity=20.0,
)

boolean_result.to_csv(
    REPORTS_DIR / "selection_boolean_filter_news.csv",
    index=False,
)

print("Boolean filter result:")
display(boolean_result.head())

# 5. isin filtering
isin_result = isin_filter(
    selection_df,
    values=["unknown", "en"],
    column="language",
    exclude=False,
)

isin_result.to_csv(
    REPORTS_DIR / "selection_isin_filter_news.csv",
    index=False,
)

print("isin filter result:")
display(isin_result.head())

# 6. isin exclusion filtering
isin_exclusion_result = isin_filter(
    selection_df,
    values=["unknown"],
    column="language",
    exclude=True,
)

isin_exclusion_result.to_csv(
    REPORTS_DIR / "selection_isin_exclusion_filter_news.csv",
    index=False,
)

print("isin exclusion result:")
display(isin_exclusion_result.head())

# 7. between filtering
between_result = between_filter(
    selection_df,
    col="rating_score",
    low=2.0,
    high=8.0,
)

between_result.to_csv(
    REPORTS_DIR / "selection_between_filter_news.csv",
    index=False,
)

print("between filter result:")
display(between_result.head())

logging.info("Lab 8 notebook Part 5 complete")


Selected columns:


,record_id,title,document_type,category,source_name,rating_score,language,content_length,published_year
0,1,"Why EOS, Humm, New Hope, and Sims shares are s...",news_api,news_api,Motley Fool Australia,1.81,unknown,181,NaN
1,2,SWDC partners FTID to boost S’West rural devt,news_api,news_api,The Punch,2.26,unknown,226,NaN
2,3,"GATE 2026 final answer key not released yet, r...",news_api,news_api,The Times of India,2.60,unknown,260,NaN
3,4,LTM Named NVIDIA Partner Network ‘Rising Star ...,news_api,news_api,Financial Post,2.60,unknown,260,NaN
4,5,"Asian stocks rally as oil retreats, Fed in spo...",news_api,news_api,CNA,2.60,unknown,260,NaN


loc filter result:


,record_id,title,document_type,category,rating_score,content_length
45,46,Untitled record,pdf,pdf,10.0,1019
47,48,Untitled record,pdf_two_column,pdf_two_column,10.0,2124
49,50,Untitled record,word,word,10.0,1080
50,51,Untitled record,word_two_column,word_two_column,10.0,1877
52,53,AI Market Growth,excel,Business,7.0,16


iloc sample result:


...

,record_id,source_name,author,title,description,url,publishedAt,source_path,fetched_at,version,...,title_length,rating_score,overview,genres,popularity,release_date,release_year,original_language,vote_average,vote_count
0,1,Motley Fool Australia,James Mickleboro,"Why EOS, Humm, New Hope, and Sims shares are s...",These shares are having a good session on hump...,https://www.fool.com.au/2026/03/18/why-eos-hum...,2026-03-18T02:40:46Z,NaN,NaT,NaN,...,66,1.81,These shares are having a good session on hump...,news_api,181.0,NaT,NaN,unknown,1.81,1.0
1,2,The Punch,Punch Newspapers,SWDC partners FTID to boost S’West rural devt,The South-West Development Commission (SWDC) a...,https://punchng.com/swdc-partners-ftid-to-boos...,2026-03-18T02:34:47Z,NaN,NaT,NaN,...,45,2.26,The South-West Development Commission (SWDC) a...,news_api,226.0,NaT,NaN,unknown,2.26,1.0
2,3,The Times of India,TOI Education,"GATE 2026 final answer key not released yet, r...",The Indian Institute of Technology Guwahati ha...,https://timesofindia.indiatimes.com/education/...,2026-03-18T02:31:14Z,NaN,NaT,NaN,...,98,2.60,The Indian Institute of Technology Guwahati ha...,news_api,260.0,NaT,NaN,unknown,2.60,1.0
3,4,Financial Post,Business Wire,LTM Named NVIDIA Partner Network ‘Rising Star ...,"MUMBAI, India — LTM, the Business Creativity p...",https://financialpost.com/pmn/business-wire-ne...,2026-03-18T02:30:15Z,NaN,NaT,NaN,...,96,2.60,"MUMBAI, India — LTM, the Business Creativity p...",news_api,260.0,NaT,NaN,unknown,2.60,1.0
4,5,CNA,NaN,"Asian stocks rally as oil retreats, Fed in spo...","SYDNEY, March 18 : Asian shares rallied on Wed...",https://www.channelnewsasia.com/business/asian...,2026-03-18T02:27:11Z,NaN,NaT,NaN,...,52,2.60,"SYDNEY, March 18 : Asian shares rallied on Wed...",news_api,260.0,NaT,NaN,unknown,2.60,1.0


isin exclusion result:


,record_id,source_name,author,title,description,url,publishedAt,source_path,fetched_at,version,...,title_length,rating_score,overview,genres,popularity,release_date,release_year,original_language,vote_average,vote_count


between filter result:


,record_id,source_name,author,title,description,url,publishedAt,source_path,fetched_at,version,...,title_length,rating_score,overview,genres,popularity,release_date,release_year,original_language,vote_average,vote_count
1,2,The Punch,Punch Newspapers,SWDC partners FTID to boost S’West rural devt,The South-West Development Commission (SWDC) a...,https://punchng.com/swdc-partners-ftid-to-boos...,2026-03-18T02:34:47Z,NaN,NaT,NaN,...,45,2.26,The South-West Development Commission (SWDC) a...,news_api,226.0,NaT,NaN,unknown,2.26,1.0
2,3,The Times of India,TOI Education,"GATE 2026 final answer key not released yet, r...",The Indian Institute of Technology Guwahati ha...,https://timesofindia.indiatimes.com/education/...,2026-03-18T02:31:14Z,NaN,NaT,NaN,...,98,2.60,The Indian Institute of Technology Guwahati ha...,news_api,260.0,NaT,NaN,unknown,2.60,1.0
3,4,Financial Post,Business Wire,LTM Named NVIDIA Partner Network ‘Rising Star ...,"MUMBAI, India — LTM, the Business Creativity p...",https://financialpost.com/pmn/business-wire-ne...,2026-03-18T02:30:15Z,NaN,NaT,NaN,...,96,2.60,"MUMBAI, India — LTM, the Business Creativity p...",news_api,260.0,NaT,NaN,unknown,2.60,1.0
4,5,CNA,NaN,"Asian stocks rally as oil retreats, Fed in spo...","SYDNEY, March 18 : Asian shares rallied on Wed...",https://www.channelnewsasia.com/business/asian...,2026-03-18T02:27:11Z,NaN,NaT,NaN,...,52,2.60,"SYDNEY, March 18 : Asian shares rallied on Wed...",news_api,260.0,NaT,NaN,unknown,2.60,1.0
5,6,The Times of India,Reuters,Trump administration defends Anthropic blackli...,Defense Secretary Pete Hegseth designated Anth...,https://economictimes.indiatimes.com/tech/tech...,2026-03-18T02:16:55Z,NaN,NaT,NaN,...,63,2.60,Defense Secretary Pete Hegseth designated Anth...,news_api,260.0,NaT,NaN,unknown,2.60,1.0


In [17]:
import re

from src.analytics.regex_ops import (
    extract_year_from_title,
    extract_any_year_from_title,
    filter_titles_starting_with,
    extract_number_from_title,
    crime_overview_count,
    crime_overview_rows,
    short_overviews,
    extract_genres,
    top_genres,
)

logging.info("Lab 8 notebook Part 6 started: regex operations")

regex_df = eda_df.copy()

regex_df["title_year_parentheses"] = extract_year_from_title(regex_df["title"])
regex_df["title_any_year"] = extract_any_year_from_title(regex_df["title"])

print("Title year extraction:")
display(regex_df[["title", "title_year_parentheses", "title_any_year"]].head())

titles_starting_with_the = filter_titles_starting_with(regex_df, prefix="The")

titles_starting_with_the.to_csv(REPORTS_DIR / "regex_titles_starting_with_the_news.csv", index=False)

print("Titles starting with 'The':")
display(titles_starting_with_the.head())

regex_df = extract_number_from_title(regex_df)

print("Title numbers:")
display(regex_df[["title", "title_number"]].head())

crime_count = crime_overview_count(regex_df)

pd.DataFrame([{
    "crime_related_content_count": crime_count,
}]).to_csv(REPORTS_DIR / "regex_crime_content_count_news.csv", index=False)

print("Crime-related content count:")
print(crime_count)

crime_rows = crime_overview_rows(regex_df)

crime_rows.to_csv(REPORTS_DIR / "regex_crime_content_rows_news.csv", index=False)

print("Crime-related rows:")
display(crime_rows.head())

short_content_rows = short_overviews(regex_df, max_chars=40)

short_content_rows.to_csv(REPORTS_DIR / "regex_short_content_news.csv", index=False)

print("Short content rows:")
display(short_content_rows.head())

regex_df = extract_genres(regex_df)

print("Extracted category/genre lists:")
display(regex_df[["title", "category", "genre_list"]].head())

top_category_labels = top_genres(regex_df, n=15)

top_category_df = pd.DataFrame(
    top_category_labels,
    columns=["category_label", "count"],
)

top_category_df.to_csv(REPORTS_DIR / "regex_top_category_labels_news.csv", index=False)

print("Top category labels:")
display(top_category_df)

record_id_pattern = re.compile(r"^\d+$")
url_pattern = re.compile(r"^https?://", re.IGNORECASE)

regex_df["is_valid_record_id"] = regex_df["record_id"].astype(str).str.match(record_id_pattern)
regex_df["has_valid_url_format"] = regex_df["url"].fillna("").astype(str).str.match(url_pattern)

regex_df.to_csv(REPORTS_DIR / "regex_processed_news_dataset.csv", index=False)

print("Record ID / URL validation:")
display(regex_df[["record_id", "title", "url", "is_valid_record_id", "has_valid_url_format"]].head())

logging.info("Lab 8 notebook Part 6 complete")

Title year extraction:


,title,title_year_parentheses,title_any_year
0,"Why EOS, Humm, New Hope, and Sims shares are s...",NaN,NaN
1,SWDC partners FTID to boost S’West rural devt,NaN,NaN
2,"GATE 2026 final answer key not released yet, r...",NaN,2026
3,LTM Named NVIDIA Partner Network ‘Rising Star ...,NaN,2026
4,"Asian stocks rally as oil retreats, Fed in spo...",NaN,NaN


Titles starting with 'The':


,record_id,source_name,author,title,description,url,publishedAt,source_path,fetched_at,version,...,overview,genres,popularity,release_date,release_year,original_language,vote_average,vote_count,title_year_parentheses,title_any_year
341,342,unknown,NaN,The King's Speech,NaN,NaN,NaN,https://www.scrapethissite.com/pages/ajax-java...,2026-04-06 00:28:38.889,1.0,...,The King's Speech,scraped_json_api,17.0,NaT,NaN,unknown,0.17,1.0,NaN,NaN
343,344,unknown,NaN,The Social Network,NaN,NaN,NaN,https://www.scrapethissite.com/pages/ajax-java...,2026-04-06 00:28:38.893,1.0,...,The Social Network,scraped_json_api,18.0,NaT,NaN,unknown,0.18,1.0,NaN,NaN
344,345,unknown,NaN,The Fighter,NaN,NaN,NaN,https://www.scrapethissite.com/pages/ajax-java...,2026-04-06 00:28:38.894,1.0,...,The Fighter,scraped_json_api,11.0,NaT,NaN,unknown,0.11,1.0,NaN,NaN
349,350,unknown,NaN,The Lost Thing,NaN,NaN,NaN,https://www.scrapethissite.com/pages/ajax-java...,2026-04-06 00:28:38.896,1.0,...,The Lost Thing,scraped_json_api,14.0,NaT,NaN,unknown,0.14,1.0,NaN,NaN
351,352,unknown,NaN,The Wolfman,NaN,NaN,NaN,https://www.scrapethissite.com/pages/ajax-java...,2026-04-06 00:28:38.897,1.0,...,The Wolfman,scraped_json_api,11.0,NaT,NaN,unknown,0.11,1.0,NaN,NaN


Title numbers:


...

,title,category,genre_list
0,"Why EOS, Humm, New Hope, and Sims shares are s...",news_api,"[news, api]"
1,SWDC partners FTID to boost S’West rural devt,news_api,"[news, api]"
2,"GATE 2026 final answer key not released yet, r...",news_api,"[news, api]"
3,LTM Named NVIDIA Partner Network ‘Rising Star ...,news_api,"[news, api]"
4,"Asian stocks rally as oil retreats, Fed in spo...",news_api,"[news, api]"


Top category labels:


,category_label,count
0,scraped,1060
1,html,625
2,json,569
3,paginated,500
4,api,465
5,word,74
6,run,56
7,pdf,46
8,news,30
9,two,27


Record ID / URL validation:


,record_id,title,url,is_valid_record_id,has_valid_url_format
0,1,"Why EOS, Humm, New Hope, and Sims shares are s...",https://www.fool.com.au/2026/03/18/why-eos-hum...,True,True
1,2,SWDC partners FTID to boost S’West rural devt,https://punchng.com/swdc-partners-ftid-to-boos...,True,True
2,3,"GATE 2026 final answer key not released yet, r...",https://timesofindia.indiatimes.com/education/...,True,True
3,4,LTM Named NVIDIA Partner Network ‘Rising Star ...,https://financialpost.com/pmn/business-wire-ne...,True,True
4,5,"Asian stocks rally as oil retreats, Fed in spo...",https://www.channelnewsasia.com/business/asian...,True,True


In [18]:
from src.analytics.quality_report import (
    missing_value_report,
    zero_as_missing,
    outlier_report,
    rating_validity_report,
    duplicate_id_report,
    title_quality_report,
    format_consistency_report,
    full_quality_report,
    save_quality_report,
    save_missing_heatmap,
)

logging.info("Lab 8 notebook Part 7 started: data quality")

quality_df_source = eda_df.copy()

missing_report = missing_value_report(quality_df_source)

missing_report.to_csv(REPORTS_DIR / "quality_missing_value_report_news.csv", index=False)

print("Missing value report:")
display(missing_report)

zero_report = zero_as_missing(
    quality_df_source,
    cols=[col for col in ["mentions", "content_length", "title_length"] if col in quality_df_source.columns],
)

zero_report.to_csv(REPORTS_DIR / "quality_zero_as_missing_report_news.csv", index=False)

print("Zero-as-missing report:")
display(zero_report)

outliers = outlier_report(quality_df_source)

outliers.to_csv(REPORTS_DIR / "quality_outlier_report_news.csv", index=False)

print("Outlier report:")
display(outliers)

rating_report = rating_validity_report(quality_df_source)

rating_report.to_csv(REPORTS_DIR / "quality_rating_validity_report_news.csv", index=False)

print("Rating validity report:")
display(rating_report)

duplicate_record_id_count = int(quality_df_source["record_id"].duplicated().sum()) if "record_id" in quality_df_source.columns else 0
duplicate_url_count = int(quality_df_source["url"].dropna().duplicated().sum()) if "url" in quality_df_source.columns else 0

news_duplicate_report = pd.DataFrame([
    {
        "column": "record_id",
        "issue": "Duplicate record IDs",
        "count": duplicate_record_id_count,
        "pct": round(duplicate_record_id_count / len(quality_df_source) * 100, 2),
        "severity": "HIGH" if duplicate_record_id_count > 0 else "LOW",
    },
    {
        "column": "url",
        "issue": "Duplicate URLs",
        "count": duplicate_url_count,
        "pct": round(duplicate_url_count / len(quality_df_source) * 100, 2),
        "severity": "MEDIUM" if duplicate_url_count > 0 else "LOW",
    },
])

news_duplicate_report.to_csv(REPORTS_DIR / "quality_duplicate_news_report.csv", index=False)

print("News duplicate report:")
display(news_duplicate_report)

generic_duplicate_report = duplicate_id_report(quality_df_source)

generic_duplicate_report.to_csv(REPORTS_DIR / "quality_duplicate_id_report_news.csv", index=False)

title_report = title_quality_report(quality_df_source)

title_report.to_csv(REPORTS_DIR / "quality_title_report_news.csv", index=False)

print("Title quality report:")
display(title_report)

format_report = format_consistency_report(quality_df_source)

format_report.to_csv(REPORTS_DIR / "quality_format_consistency_report_news.csv", index=False)

print("Format consistency report:")
display(format_report)

full_report = full_quality_report(quality_df_source)

full_report = pd.concat([full_report, news_duplicate_report], ignore_index=True)

save_quality_report(
    full_report,
    output_path=str(REPORTS_DIR / "full_quality_report_news.csv"),
)

save_quality_report(
    full_report,
    output_path=str(REPORTS_DIR / "full_quality_report.csv"),
)

print("Full quality report:")
display(full_report)

save_missing_heatmap(
    quality_df_source,
    output_path=str(CHARTS_DIR / "missing_values_heatmap.png"),
)

print("Saved quality report to:")
print(REPORTS_DIR / "full_quality_report_news.csv")

print("Saved missing value heatmap to:")
print(CHARTS_DIR / "missing_values_heatmap.png")

logging.info("Lab 8 notebook Part 7 complete")

Missing value report:


,column,missing_count,missing_pct,severity
0,underline,1455,100.00,HIGH
1,preview_text,1448,99.52,HIGH
2,Total Articles,1446,99.38,HIGH
3,Metric,1446,99.38,HIGH
4,Politics Articles,1446,99.38,HIGH
5,Business Articles,1446,99.38,HIGH
6,Average Sentiment,1446,99.38,HIGH
7,Highest Mentions,1446,99.38,HIGH
8,Total Mentions,1446,99.38,HIGH
9,Average Mentions,1446,99.38,HIGH


Zero-as-missing report:


,column,issue,count,pct,severity
0,mentions,Zero values may represent missing or incomplet...,1365,93.81,HIGH


Outlier report:


...

Format consistency report:


""


Full quality report:


,column,issue,count,pct,severity
0,underline,Missing values,1455,100.00,HIGH
1,preview_text,Missing values,1448,99.52,HIGH
2,Total Articles,Missing values,1446,99.38,HIGH
3,Metric,Missing values,1446,99.38,HIGH
4,Politics Articles,Missing values,1446,99.38,HIGH
...,...,...,...,...,...
56,file_name,Missing values,30,2.06,LOW
57,title,Unusually short titles,30,2.06,LOW
58,wins,IQR outliers detected,10,1.60,LOW
59,record_id,Duplicate record IDs,0,0.00,LOW


Saved quality report to:
data\processed\analytics\reports\full_quality_report_news.csv
Saved missing value heatmap to:
data\processed\analytics\charts\missing_values_heatmap.png


In [7]:
from googleapiclient.http import MediaFileUpload

from src.utils.upload_utils import authenticate_drive, FOLDER_ID

logging.info("Lab 8 notebook Part 8 started: Google Drive upload")

AMILA_EMAIL = os.getenv("AMILA_EMAIL") or "amila.causevic@ibu.edu.ba"

chart_paths = [
    CHARTS_DIR / "rating_score_distribution.png",
    CHARTS_DIR / "rating_distribution.png",
    CHARTS_DIR / "popularity_distribution.png",
    CHARTS_DIR / "mentions_distribution.png",
    CHARTS_DIR / "content_length_distribution.png",
    CHARTS_DIR / "language_distribution.png",
    CHARTS_DIR / "published_year_distribution.png",
    CHARTS_DIR / "release_year_distribution.png",
    CHARTS_DIR / "category_distribution.png",
    CHARTS_DIR / "genre_distribution.png",
    CHARTS_DIR / "document_type_distribution.png",
    CHARTS_DIR / "source_distribution.png",
    CHARTS_DIR / "missing_values_heatmap.png",
]

service = authenticate_drive()

upload_results = []

for chart_path in chart_paths:
    chart_path = Path(chart_path)

    if not chart_path.exists():
        print("Missing chart, skipping:", chart_path)
        continue

    file_metadata = {
        "name": chart_path.name,
    }

    if FOLDER_ID:
        file_metadata["parents"] = [FOLDER_ID]

    media = MediaFileUpload(
        str(chart_path),
        mimetype="image/png",
        resumable=False,
    )

    uploaded_file = service.files().create(
        body=file_metadata,
        media_body=media,
        fields="id, webViewLink",
    ).execute()

    file_id = uploaded_file.get("id")
    file_url = uploaded_file.get("webViewLink")

    if AMILA_EMAIL:
        service.permissions().create(
            fileId=file_id,
            body={
                "type": "user",
                "role": "reader",
                "emailAddress": AMILA_EMAIL,
            },
            sendNotificationEmail=False,
        ).execute()

    upload_results.append({
        "file_name": chart_path.name,
        "local_path": str(chart_path),
        "drive_file_id": file_id,
        "drive_url": file_url,
        "shared_with": AMILA_EMAIL,
    })

    print("Uploaded and shared:", chart_path.name, "->", file_url)

upload_report = pd.DataFrame(upload_results)

upload_report.to_csv(
    REPORTS_DIR / "google_drive_chart_uploads_news.csv",
    index=False,
)

upload_report.to_csv(
    REPORTS_DIR / "google_drive_chart_uploads.csv",
    index=False,
)

display(upload_report)

logging.info("Lab 8 notebook Part 8 complete: charts uploaded and shared")


Uploaded and shared: rating_score_distribution.png -> https://drive.google.com/file/d/1lemgSTGtPFjloo1IlPXHz_gnT1j8oQ87/view?usp=drivesdk
Uploaded and shared: rating_distribution.png -> https://drive.google.com/file/d/1tzLQDD55Eb1FeR5baBwf6VnoEy1P0Ta1/view?usp=drivesdk
Uploaded and shared: popularity_distribution.png -> https://drive.google.com/file/d/1wjW1TNFg-nvcVDz3aICxUkwi1LNTUMYo/view?usp=drivesdk
Uploaded and shared: mentions_distribution.png -> https://drive.google.com/file/d/1hJk1WoRsVG97cesgrBi2hRIL9xksR-AK/view?usp=drivesdk
Uploaded and shared: content_length_distribution.png -> https://drive.google.com/file/d/1XXssB1bgaTZTvvBmMTiDSE5oY7XxbvHh/view?usp=drivesdk
Uploaded and shared: language_distribution.png -> https://drive.google.com/file/d/1NgN11w797fdLqyHB5febEaSD7cBNJBb7/view?usp=drivesdk
Uploaded and shared: published_year_distribution.png -> https://drive.google.com/file/d/1alnn2p9zBoaisbFVOx1LOnQtwSxyNu87/view?usp=drivesdk
Uploaded and shared: release_year_distribution

,file_name,local_path,drive_file_id,drive_url,shared_with
0,rating_score_distribution.png,data\processed\analytics\charts\rating_score_d...,1lemgSTGtPFjloo1IlPXHz_gnT1j8oQ87,https://drive.google.com/file/d/1lemgSTGtPFjlo...,amila.causevic@ibu.edu.ba
1,rating_distribution.png,data\processed\analytics\charts\rating_distrib...,1tzLQDD55Eb1FeR5baBwf6VnoEy1P0Ta1,https://drive.google.com/file/d/1tzLQDD55Eb1Fe...,amila.causevic@ibu.edu.ba
2,popularity_distribution.png,data\processed\analytics\charts\popularity_dis...,1wjW1TNFg-nvcVDz3aICxUkwi1LNTUMYo,https://drive.google.com/file/d/1wjW1TNFg-nvcV...,amila.causevic@ibu.edu.ba
3,mentions_distribution.png,data\processed\analytics\charts\mentions_distr...,1hJk1WoRsVG97cesgrBi2hRIL9xksR-AK,https://drive.google.com/file/d/1hJk1WoRsVG97c...,amila.causevic@ibu.edu.ba
4,content_length_distribution.png,data\processed\analytics\charts\content_length...,1XXssB1bgaTZTvvBmMTiDSE5oY7XxbvHh,https://drive.google.com/file/d/1XXssB1bgaTZTv...,amila.causevic@ibu.edu.ba
5,language_distribution.png,data\processed\analytics\charts\language_distr...,1NgN11w797fdLqyHB5febEaSD7cBNJBb7,https://drive.google.com/file/d/1NgN11w797fdLq...,amila.causevic@ibu.edu.ba
6,published_year_distribution.png,data\processed\analytics\charts\published_year...,1alnn2p9zBoaisbFVOx1LOnQtwSxyNu87,https://drive.google.com/file/d/1alnn2p9zBoais...,amila.causevic@ibu.edu.ba
7,release_year_distribution.png,data\processed\analytics\charts\release_year_d...,1ZPRyOipswFXLY8G6PfQE0GjpdT12u3AM,https://drive.google.com/file/d/1ZPRyOipswFXLY...,amila.causevic@ibu.edu.ba
8,category_distribution.png,data\processed\analytics\charts\category_distr...,1kUG0XYxALnWQgiS2u2XN_l4ME4LUeVR6,https://drive.google.com/file/d/1kUG0XYxALnWQg...,amila.causevic@ibu.edu.ba
9,genre_distribution.png,data\processed\analytics\charts\genre_distribu...,17mdbZt4fOV4EesGe1zeohh7AZzBLlOf3,https://drive.google.com/file/d/17mdbZt4fOV4Ee...,amila.causevic@ibu.edu.ba
